In [ ]:
!pip install -q faiss-cpu sentence-transformers sqlalchemy pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 28.3 MB/s eta 0:00:00


In [ ]:
import faiss
import pandas as pd
import sqlite3
import numpy as np
from sqlalchemy import create_engine
from sentence_transformers import SentenceTransformer

print("All imports successful")

All imports successful


In [ ]:
conn = sqlite3.connect("customer_support.db")
cursor = conn.cursor()

In [ ]:
# Create orders table
cursor.execute("""
CREATE TABLE IF NOT EXISTS orders (
    order_id INTEGER PRIMARY KEY,
    customer_name TEXT,
    product_name TEXT,
    order_status TEXT,
    delivery_date TEXT
)
""")

# Create payments table

cursor.execute("""
CREATE TABLE IF NOT EXISTS payments (
    payment_id INTEGER PRIMARY KEY,
    order_id INTEGER,
    payment_status TEXT,
    amount REAL,
    payment_method TEXT
)
""")

# Create refunds table

cursor.execute("""
CREATE TABLE IF NOT EXISTS refunds (
    refund_id INTEGER PRIMARY KEY,
    order_id INTEGER,
    refund_status TEXT,
    refund_amount REAL
)
""")


# Clear old data if any
cursor.execute("DELETE FROM orders")
cursor.execute("DELETE FROM payments")
cursor.execute("DELETE FROM refunds")

# Insert sample data
orders_data = [
    (101, "Rahul", "iPhone 15", "Shipped", "2026-03-18"),
    (102, "Rahul", "Boat Headphones", "Delivered", "2026-03-10"),
    (103, "Priya", "Samsung TV", "Processing", "2026-03-20")
]


payments_data = [
    (1, 101, "Paid", 79999, "Credit Card"),
    (2, 102, "Paid", 2499, "UPI"),
    (3, 103, "Pending", 45999, "Debit Card")
]


refunds_data = [
    (1, 102, "Refunded", 2499)
]

cursor.executemany("INSERT INTO orders VALUES (?, ?, ?, ?, ?)", orders_data)
cursor.executemany("INSERT INTO payments VALUES (?, ?, ?, ?, ?)", payments_data)
cursor.executemany("INSERT INTO refunds VALUES (?, ?, ?, ?)", refunds_data)


conn.commit()
conn.close()

print("Database created successfully")





Database created successfully


In [ ]:
policy_docs = [
    "Return Policy: Customers can return products within 7 days of delivery. The product must be unused and in original packaging. Refund is processed within 5 to 7 business days after approval.",

    "Shipping Policy: Orders are usually shipped within 2 business days. Customers receive tracking updates once the order is dispatched. Delivery timelines vary depending on the location.",

    "Payment Policy: Payments can be made using Credit Card, Debit Card, UPI, and Net Banking. If payment fails, users can retry using another payment method."
]

print("Policy documents ready")

Policy documents ready


In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

doc_embeddings = model.encode(policy_docs)
doc_embeddings = np.array(doc_embeddings).astype("float32")

index = faiss.IndexFlatL2(doc_embeddings.shape[1])
index.add(doc_embeddings)

print("FAISS index created successfully")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS index created successfully


In [ ]:
def retrieve_policy(query, top_k=2):
    query_embedding = model.encode([query])
    query_embedding = np.array(query_embedding).astype("float32")

    distances, indices = index.search(query_embedding, top_k)
    results = [policy_docs[i] for i in indices[0]]

    return results

In [ ]:
def run_sql(query):
    conn = sqlite3.connect("customer_support.db")
    df = pd.read_sql_query(query, conn)
    conn.close()
    return df

In [ ]:
def customer_support_agent(user_query):
    q = user_query.lower()
    response_parts = []

    # SQL logic
    if "order 101" in q:
        order_df = run_sql("SELECT * FROM orders WHERE order_id = 101")
        payment_df = run_sql("SELECT * FROM payments WHERE order_id = 101")

        response_parts.append("Order Details:\n" + order_df.to_string(index=False))
        response_parts.append("Payment Details:\n" + payment_df.to_string(index=False))

    elif "order 102" in q:
        order_df = run_sql("SELECT * FROM orders WHERE order_id = 102")
        payment_df = run_sql("SELECT * FROM payments WHERE order_id = 102")
        refund_df = run_sql("SELECT * FROM refunds WHERE order_id = 102")

        response_parts.append("Order Details:\n" + order_df.to_string(index=False))
        response_parts.append("Payment Details:\n" + payment_df.to_string(index=False))
        response_parts.append("Refund Details:\n" + refund_df.to_string(index=False))

    elif "rahul" in q:
        customer_orders = run_sql("SELECT * FROM orders WHERE customer_name = 'Rahul'")
        response_parts.append("Customer Orders:\n" + customer_orders.to_string(index=False))

    # RAG logic
    if any(word in q for word in ["policy", "return", "refund process", "shipping", "payment method"]):
        retrieved_docs = retrieve_policy(user_query, top_k=2)
        response_parts.append("Policy Information:\n" + "\n\n".join(retrieved_docs))

    if not response_parts:
        return "Sorry, I could not find relevant information for your query."

    return "\n\n".join(response_parts)


In [ ]:
query = "What is the refund status for order 102 and what is the return process?"
print(customer_support_agent(query))

Order Details:
 order_id customer_name    product_name order_status delivery_date
      102         Rahul Boat Headphones    Delivered    2026-03-10

Payment Details:
 payment_id  order_id payment_status  amount payment_method
          2       102           Paid  2499.0            UPI

Refund Details:
 refund_id  order_id refund_status  refund_amount
         1       102      Refunded         2499.0

Policy Information:
Return Policy: Customers can return products within 7 days of delivery. The product must be unused and in original packaging. Refund is processed within 5 to 7 business days after approval.

Payment Policy: Payments can be made using Credit Card, Debit Card, UPI, and Net Banking. If payment fails, users can retry using another payment method.


In [ ]:
query2 = "Where is my order 101 and what is the shipping policy?"
print(customer_support_agent(query2))

Order Details:
 order_id customer_name product_name order_status delivery_date
      101         Rahul    iPhone 15      Shipped    2026-03-18

Payment Details:
 payment_id  order_id payment_status  amount payment_method
          1       101           Paid 79999.0    Credit Card

Policy Information:
Shipping Policy: Orders are usually shipped within 2 business days. Customers receive tracking updates once the order is dispatched. Delivery timelines vary depending on the location.

Return Policy: Customers can return products within 7 days of delivery. The product must be unused and in original packaging. Refund is processed within 5 to 7 business days after approval.


In [ ]:
import re

def chatbot_agent(user_query):
    q = user_query.lower()
    response_parts = []

    # Extract order id
    order_match = re.search(r'order\s*(\d+)', q)
    order_id = order_match.group(1) if order_match else None

    # Get all customer names from DB
    customers_df = run_sql("SELECT DISTINCT customer_name FROM orders")
    customer_list = [name.lower() for name in customers_df["customer_name"].tolist()]

    found_customer = None
    for name in customer_list:
        if name in q:
            found_customer = name
            break

    # Query by order id
    if order_id:
        order_df = run_sql(f"SELECT * FROM orders WHERE order_id = {order_id}")
        payment_df = run_sql(f"SELECT * FROM payments WHERE order_id = {order_id}")
        refund_df = run_sql(f"SELECT * FROM refunds WHERE order_id = {order_id}")

        if not order_df.empty:
            row = order_df.iloc[0]
            response_parts.append(
                f"Order {order_id} for {row['customer_name']} ({row['product_name']}) is currently '{row['order_status']}'. Delivery date is {row['delivery_date']}."
            )

        if not payment_df.empty:
            row = payment_df.iloc[0]
            response_parts.append(
                f"Payment status is '{row['payment_status']}'. Amount: ₹{int(row['amount'])} paid via {row['payment_method']}."
            )

        if not refund_df.empty:
            row = refund_df.iloc[0]
            response_parts.append(
                f"Refund status is '{row['refund_status']}'. Refund amount: ₹{int(row['refund_amount'])}."
            )

    # Query by customer name
    elif found_customer:
        customer_orders = run_sql(
            f"SELECT * FROM orders WHERE LOWER(customer_name) = LOWER('{found_customer}')"
        )

        if not customer_orders.empty:
            response_parts.append(
                f"Here are the orders for {found_customer.capitalize()}:\n\n" +
                customer_orders.to_string(index=False)
            )

    # Policy retrieval
    if any(word in q for word in ["policy", "return", "refund process", "shipping", "payment method"]):
        retrieved_docs = retrieve_policy(user_query, top_k=2)
        response_parts.append(
            "Relevant policy information:\n\n" + "\n\n".join(retrieved_docs)
        )

    if not response_parts:
        return "Sorry, I could not find relevant information."

    return "\n\n".join(response_parts)

In [ ]:
def start_chatbot():
    print("Customer Support Chatbot is ready.")
    print("Type 'exit' to stop.\n")

    while True:
        user_query = input("You: ")

        if user_query.lower() == "exit":
            print("Bot: Thank you for chatting. Goodbye!")
            break

        response = chatbot_agent(user_query)
        print("\nBot:", response)
        print("-" * 80)

start_chatbot()

Customer Support Chatbot is ready.
Type 'exit' to stop.

You: show priya order

Bot: Here are the orders for Priya:

 order_id customer_name product_name order_status delivery_date
      103         Priya   Samsung TV   Processing    2026-03-20
--------------------------------------------------------------------------------
You: exit
Bot: Thank you for chatting. Goodbye!


In [ ]:
chat_history = []

def chatbot_with_memory():
    print("Customer Support Chatbot with Memory is ready.")
    print("Type 'exit' to stop.\n")

    while True:
        user_query = input("You: ")

        if user_query.lower() == "exit":
            print("Bot: Thank you for chatting. Goodbye!")
            break

        response = chatbot_agent(user_query)

        chat_history.append({"user": user_query, "bot": response})

        print("\nBot:", response)
        print("\nChat History:")
        for i, chat in enumerate(chat_history, 1):
            print(f"{i}. You: {chat['user']}")
            print(f"   Bot: {chat['bot']}")
        print("-" * 80)

chatbot_with_memory()

Customer Support Chatbot with Memory is ready.
Type 'exit' to stop.

You: Show Rahul orders

Bot: Rahul has the following orders:
 order_id customer_name    product_name order_status delivery_date
      101         Rahul       iPhone 15      Shipped    2026-03-18
      102         Rahul Boat Headphones    Delivered    2026-03-10

Chat History:
1. You: Show Rahul orders
   Bot: Rahul has the following orders:
 order_id customer_name    product_name order_status delivery_date
      101         Rahul       iPhone 15      Shipped    2026-03-18
      102         Rahul Boat Headphones    Delivered    2026-03-10
--------------------------------------------------------------------------------
You: exit
Bot: Thank you for chatting. Goodbye!


In [ ]:
!pip install -q gradio

In [ ]:
import gradio as gr

def gradio_chat(user_message, history):
    bot_response = chatbot_agent(user_message)
    history.append((user_message, bot_response))
    return "", history

with gr.Blocks() as demo:
    gr.Markdown("# Customer Support Chatbot")
    gr.Markdown("Ask about orders, payments, refunds, return policy, shipping policy, etc.")

    chatbot = gr.Chatbot()
    msg = gr.Textbox(placeholder="Type your question here...")
    clear = gr.Button("Clear Chat")

    msg.submit(gradio_chat, inputs=[msg, chatbot], outputs=[msg, chatbot])
    clear.click(lambda: [], None, chatbot)

demo.launch(debug=True)

/tmp/ipykernel_732/2042850573.py:12: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot()
/tmp/ipykernel_732/2042850573.py:12: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://6e0f40f2e78ef6bcdd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://6e0f40f2e78ef6bcdd.gradio.live
